# Module 6 Data Cleaning Checklist Answers

This notebook-style document answers the 14 required data-cleaning checklist questions for the project. It references the cleaning files in `Python-Group-Project/data-cleaning`.

## 1. State The Project Vision And Question

**Status: Done**

**Project vision:** Financial Agent Deployment. The broader project explores how financial decision-makers can trust AI agents in real-world business workflows, especially for equity research or financial-report analysis.

**Business question:** For eight large U.S. technology companies, how can daily stock prices, company news, and SEC filing events be combined into a dashboard-ready dataset that helps users compare market movement with related news and filing context?

**Main metric or comparison:** The main metric is daily stock movement, especially `daily_return`, compared across `symbol`, `date`, news availability, and filing-event flags.

**Evidence or explanation:** See `README.md` Data Source section, `plan.md`, and the cleaned dashboard output `data/processed/dashboard_daily.csv`.

## 2. Define One Row

**Status: Done**

**Prices:** One row represents one company ticker on one trading date. Unique key: `symbol` + `date`.

**News:** One row represents one company ticker on one news date. Unique key: `symbol` + `date`.

**Filings:** One row represents one company ticker, one filing date, and one filing type. Unique key: `symbol` + `date` + `document_type`.

**Dashboard:** One row represents one company ticker on one trading date, using prices as the base table. Unique key: `symbol` + `date`.

**Evidence or explanation:** See `plan.md`, `clean_data.py`, and duplicate-key checks in `cleaning.ipynb`.

## 3. Preserve The Source

**Status: Done**

**Data source:** TheFinAI/Herculean dataset on Hugging Face: https://huggingface.co/datasets/TheFinAI/Herculean

**Version/download date:** Local raw snapshot downloaded into this project workspace on October 2, 2026. The notebook also records the Hugging Face dataset revision and last-modified date when data is downloaded.

**Source preservation:** Original raw parquet files are kept unchanged in `data/raw/data`. Cleaned outputs are written separately to `data/processed`.

**How to load:** Use `pd.read_parquet('data/raw/data/prices.parquet')`, `pd.read_parquet('data/raw/data/news.parquet')`, and `pd.read_parquet('data/raw/data/filings.parquet')`.

**Evidence or explanation:** See `README.md`, `cleaning.ipynb`, and raw files in `data/raw/data`.

## 4. Inspect Columns And Types

**Status: Done**

The raw `date` fields were object/string-like and were converted to datetime. Price columns were validated as numeric: `open`, `high`, `low`, `close`, `adj_close`, and `volume`. The script also validates document categories and text columns.

**Failed conversions:** The cleaning report shows 0 changed rows for date and numeric conversions in the current snapshot, meaning all values converted successfully.

**Handling:** Invalid dates, invalid symbols, invalid prices, and unknown document types are flagged with `flag_` columns before any removal. In the current data, suspicious flag counts are 0.

**Evidence or explanation:** See `clean_data.py`, `plan.md`, `cleaning.ipynb`, and `data/processed/cleaning_report.json`.

## 5. Standardize Text

**Status: Done**

Text fields cleaned: `news.highlights`, `filings.mda_content`, and `filings.risk_content`. Cleaning included HTML unescaping, smart punctuation normalization, repeated whitespace collapse, and trimming. The cleaning is readability/encoding cleanup only; it does not rewrite the meaning.

**Before/after evidence:** The raw text includes HTML entities such as `&#8217;` and smart punctuation. The cleaned output converts these to readable punctuation and collapses extra whitespace.

**Changed rows:** `highlights` changed in 3,882 news rows. `mda_content` and `risk_content` changed in all 73 filing rows.

**IDs/leading zeros:** ID columns are kept as identifiers and are not text-standardized. There are no meaningful leading-zero ID strings in this dataset.

**Evidence or explanation:** See `clean_data.py`, before/after row previews in `cleaning.ipynb`, and `cleaning_report.json`.

## 6. Investigate Duplicates

**Status: Done**

Exact duplicate rows and duplicate business keys were checked separately.

**Results:**

- Prices duplicate key rows (`symbol`, `date`): 0
- News duplicate key rows (`symbol`, `date`): 0
- Filings duplicate key rows (`symbol`, `date`, `document_type`): 0
- Dashboard duplicate key rows (`symbol`, `date`): 0
- Full duplicate rows after cleaning: 0

**Handling:** No duplicate rows were removed because none were found. Duplicate flags remain in the cleaned files to document the check.

**Evidence or explanation:** See duplicate section in `cleaning.ipynb` and `cleaning_report.json`.

## 7. Handle Missing Data

**Status: Done**

Raw files had no missing values in the inspected columns. Cleaned source tables also have no missing values in required fields.

The dashboard has 8 missing values in `daily_return` and 8 missing values in `dollar_change`. These are expected because each of the 8 symbols has a first trading row with no prior trading day for comparison.

No values were filled or dropped. Expected derived missing values were kept to avoid inventing returns where no prior record exists.

**Evidence or explanation:** See missing-value checks in `cleaning.ipynb` and `cleaning_report.json`.

## 8. Check Suspicious Values

**Status: Done**

The script checks invalid symbols, invalid dates, duplicate keys, invalid prices, empty text, and unknown filing document types. Price sanity checks include positive prices, non-negative volume, and `high >= low`.

**Results:** Current suspicious flag counts are 0 for prices, news, filings, and dashboard.

**Handling:** Suspicious values are flagged before removal. No suspicious rows were corrected or removed in the current snapshot.

**Evidence or explanation:** See flag columns in cleaned outputs, `clean_data.py`, and `cleaning_report.json`.

## 9. Verify Merges

**Status: Done**

The dashboard uses cleaned prices as the base table because prices define trading days. News and filings are left-joined by `symbol` and `date`.

**Expected relationship:** One dashboard row per price row. News is expected for each trading date in this dataset. Filings are event-based and are not expected every trading day.

**Merge results:**

- Price rows used as base: 2,656
- Dashboard rows after merge: 2,656
- Rows with news: 2,656
- Rows without news: 0
- Rows with filings: 41
- Rows without filings: 2,615
- Unexpected extra dashboard rows: 0

**Evidence or explanation:** See `dashboard_daily` merge report in `cleaning_report.json`.

## 10. Reconcile Changes

**Status: Done**

Starting and final row counts are unchanged for the cleaned source tables.

| Dataset | Starting rows | Final rows | Difference |
| --- | ---: | ---: | ---: |
| Prices | 2,656 | 2,656 | 0 |
| News | 3,888 | 3,888 | 0 |
| Filings | 73 | 73 | 0 |
| Dashboard | 2,656 price rows | 2,656 dashboard rows | 0 |

No rows were removed. Important totals such as number of price rows and dashboard trading-day rows are preserved. The merge adds news and filing context without changing the number of price observations.

**Evidence or explanation:** See console output from `python clean_data.py`, `cleaning.ipynb`, and `cleaning_report.json`.

## 11. Check Calculations

**Status: Done**

Derived calculations include `daily_return`, `dollar_change`, `intraday_range_pct`, `trading_year`, `trading_month`, text-length fields, news counts, filing counts, and boolean flags.

`daily_return` and `dollar_change` are calculated within each `symbol` group using prior `adj_close`. The first row for each of the 8 symbols has missing return/change because there is no previous trading day. These rows are kept and excluded naturally from averages that ignore missing values.

Metric record counts: dashboard has 2,656 rows; `daily_return` and `dollar_change` each have 2,648 non-missing values. Missing group keys are flagged through symbol/date validation before calculations.

**Evidence or explanation:** See `clean_data.py`, numeric summaries in `cleaning.ipynb`, and `cleaning_report.json`.

## 12. Verify Individual Records

**Status: Done**

Five record checks are included in `cleaning.ipynb`. They cover:

1. AAPL second price row: verifies date conversion and `daily_return`.
2. First news row: verifies text normalization and positive `highlight_length`.
3. First filing row: verifies `document_type` and text-length fields.
4. Dashboard row with filing: verifies `has_filing`, `filing_count`, and `filing_types`.
5. Dashboard row without filing: verifies that no-filing trading days are preserved, not treated as bad data.

Each record check shows the original value, expected result, actual result, and whether it matches.

**Evidence or explanation:** See the `Five Record Checks` section in `cleaning.ipynb`.

## 13. Test Reproducibility

**Status: Done**

Reproducibility steps:

```powershell
cd Python-Group-Project/data-cleaning
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
python clean_data.py
jupyter notebook cleaning.ipynb
```

`clean_data.py` recreates the cleaned CSV/parquet outputs and `cleaning_report.json` from the raw parquet files. `cleaning.ipynb` can then be restarted and run from the top to validate the submitted results.

**Evidence or explanation:** See `README.md`, `requirements.txt`, `clean_data.py`, and `cleaning.ipynb`.

## 14. Document Remaining Limitations

**Status: Done**

Remaining limitations:

- The dataset is limited to eight large U.S. technology companies, so conclusions should not be generalized to the whole market.
- Filing rows are event-based and sparse; only 41 dashboard trading rows have same-day filing flags.
- News highlights are summaries, not complete original news articles.
- The cleaning validates structure, dates, symbols, prices, text fields, and joins, but it does not prove that every news or filing excerpt is substantively complete.
- Daily returns are based on available adjusted close prices and do not include broader market benchmarks unless added later.

These limitations mean the cleaned data is appropriate for a project dashboard comparing stock movement with available news and filing context, but deeper investment conclusions would require additional validation and possibly more data sources.

**Evidence or explanation:** See `README.md`, `cleaning.ipynb`, and project PDF/background document.

## Cleaning Decision Log

| Rule | Reason | Check |
| --- | --- | --- |
| Keep original raw files in `data/raw/data` and write cleaned files to `data/processed` | Preserves source data and makes the workflow reproducible | Confirm raw parquet files still exist and cleaned files are separate in `data/processed` |
| Convert `date` fields to datetime | Dates are needed for sorting, joins, and time-series dashboard views | `cleaning_report.json` shows date conversion; invalid dates are flagged with `flag_invalid_date`; current invalid date count is 0 |
| Standardize `symbol` by trimming and uppercasing | Ensures ticker joins work consistently across prices, news, filings | `flag_invalid_symbol` checks symbols against `AAPL`, `ADBE`, `AMZN`, `GOOGL`, `META`, `MSFT`, `NVDA`, `TSLA`; current invalid symbol count is 0 |
| Convert price columns to numeric and validate price logic | Dashboard metrics require numeric prices and plausible price records | Validate positive `open`, `high`, `low`, `close`, `adj_close`, non-negative `volume`, and `high >= low`; current `flag_invalid_price` count is 0 |
| Preserve price and volume units | Avoids changing the real-world meaning of stock prices and share counts | `cleaning_report.json` records unit change as `none` for price and volume columns |
| Normalize news and filing text by HTML-unescaping, punctuation normalization, whitespace collapse, and trimming | Improves readability without changing meaning | Before/after row previews in `cleaning.ipynb`; `highlight_length`, `mda_content_length`, and `risk_content_length` are positive |
| Standardize `document_type` by trimming and uppercasing | Keeps filing categories consistent for validation and grouping | Validate document type is `10-K` or `10-Q`; current `flag_unknown_document_type` count is 0 |
| Do not silently drop rows | Assignment requires explaining removals and preserving source meaning | `cleaning_report.json` reports 0 removed rows for prices, news, filings, and dashboard |
| Use prices as dashboard base table and left-join news/filings | Prices define trading days; filings are sparse event data and should not remove price rows | Dashboard row count remains 2,656; rows with filings = 41; rows without filings = 2,615 |
| Keep first-day missing `daily_return` and `dollar_change` | First trading day per symbol has no prior price, so calculated return is undefined | Missing values are exactly 8 for each field, one per symbol; not filled or dropped |


## Five Record Checks For PDF

These are the five record checks to include in the Gradescope PDF. The executable version also appears in `cleaning.ipynb`.

| # | Record | Original | Expected | Actual | Result |
| --- | --- | --- | --- | --- | --- |
| 1 | AAPL second price row | Raw AAPL price row with string/object date and adjusted close | Symbol stays `AAPL`; date converts to datetime; `daily_return` is calculated from prior AAPL adjusted close | Cleaned row has `symbol = AAPL`, valid datetime date, non-missing `daily_return`, and `flag_invalid_price = False` | Matches |
| 2 | First news row | Raw news text contains encoding/punctuation/whitespace issues | Text is normalized for readability; content meaning is preserved; `highlight_length` is positive | Cleaned row has readable `highlights`, positive `highlight_length`, and `flag_empty_text = False` | Matches |
| 3 | First filing row | Raw filing has `document_type`, `mda_content`, and `risk_content` | `document_type` remains valid (`10-K` or `10-Q`); text is normalized; length fields are positive | Cleaned row has valid `document_type`, positive `mda_content_length`, positive `risk_content_length`, and no unknown document flag | Matches |
| 4 | Dashboard row with filing | Matching price/news/filing records by `symbol` and `date` | Filing event appears in dashboard without changing price row count | Dashboard row has `has_filing = True`, `filing_count > 0`, and populated `filing_types` | Matches |
| 5 | Dashboard row without filing | Trading-day price/news row without same-day filing | Row is preserved because filings are event-based and not expected every trading day | Dashboard row has `has_filing = False`, `filing_count = 0`, and blank `filing_types` | Matches |


## How I Checked One Codex Suggestion

I used Codex to help write the governed cleaning script, but I checked its suggestions against the data and the assignment rules before accepting them.

**Codex suggestion checked:** Use cleaned prices as the base table for the dashboard and left-join news and filings by `symbol` and `date`.

**Why this needed checking:** A wrong merge strategy could accidentally drop trading days or create duplicate dashboard rows. Filings are sparse event records, so an inner join would incorrectly remove most trading days without filings.

**How I checked it:**

1. I inspected the raw data row counts: prices = 2,656 rows, news = 3,888 rows, filings = 73 rows.
2. I verified the intended row definition: dashboard should be one row per `symbol` and trading `date`, so prices should define the dashboard base.
3. I checked the merge report in `data/processed/cleaning_report.json`.
4. I confirmed the dashboard still has 2,656 rows and 0 duplicate `symbol`/`date` rows.
5. I confirmed filing absence is preserved correctly: 41 rows have same-day filings and 2,615 rows do not.

**Conclusion:** The Codex suggestion was accepted because the validation showed it preserved the real-world phenomenon: every trading-day price observation remains in the dashboard, while news and filings are added as context rather than used to filter out price rows.


**Branch Name:** Olivia-work
**Branch URL:** https://github.com/xunyangy-afk/Python-Group-Project.git
**Commit ID:** 7f32cdc